# 02 — EDA & Topic Modeling

Loads the cleaned Conti dataset from notebook 01, runs BERTopic to discover operational vocabulary clusters, and explores language distribution and message timelines.

**GPU note:** BERTopic uses Sentence-BERT for embeddings — the most expensive step. Embeddings run on GPU automatically if CUDA is available.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

PROCESSED_DIR = Path('../data/processed')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Load cleaned data

In [ ]:
df = pd.read_parquet(PROCESSED_DIR / 'conti_cleaned.parquet')
print(f'Loaded {len(df):,} messages')
print(df[['sender', 'timestamp', 'lang', 'body_len']].describe())

## 2. EDA — language, senders, timeline

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Language distribution
lang_counts = df['lang'].value_counts().head(8)
axes[0].bar(lang_counts.index, lang_counts.values)
axes[0].set_title('Language distribution')
axes[0].set_xlabel('Language')
axes[0].set_ylabel('Messages')

# Top senders
sender_counts = df['sender'].value_counts().head(15)
axes[1].barh(sender_counts.index[::-1], sender_counts.values[::-1])
axes[1].set_title('Top 15 senders')
axes[1].set_xlabel('Messages')

# Message length distribution
axes[2].hist(df['body_len'].clip(upper=1000), bins=60, edgecolor='none')
axes[2].set_title('Message length distribution')
axes[2].set_xlabel('Characters')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'eda_overview.png', dpi=150)
plt.show()

In [ ]:
# Timeline by language
if df['timestamp'].notna().sum() > 0:
    df_ts = df.set_index('timestamp').sort_index()
    fig, ax = plt.subplots(figsize=(13, 4))
    for lang in ['ru', 'en', 'unknown']:
        subset = df_ts[df_ts['lang'] == lang]['body_clean'].resample('W').count()
        if len(subset) > 0:
            ax.plot(subset.index, subset.values, label=lang, linewidth=1.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    ax.set_title('Weekly message volume by language')
    ax.set_ylabel('Messages per week')
    ax.legend()
    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / 'timeline_by_lang.png', dpi=150)
    plt.show()

## 3. Prepare corpus for BERTopic

BERTopic works best on English text. We'll run separate models for Russian and English to capture both operational vocabularies.

In [ ]:
# Separate English and Russian corpora; filter very short messages
df_en = df[(df['lang'] == 'en') & (df['body_len'] >= 20)].copy()
df_ru = df[(df['lang'] == 'ru') & (df['body_len'] >= 20)].copy()

print(f'English messages: {len(df_en):,}')
print(f'Russian messages: {len(df_ru):,}')

# Cap at 20k each to keep VRAM comfortable on 8GB
MAX_DOCS = 20_000
docs_en = df_en['body_clean'].sample(min(MAX_DOCS, len(df_en)), random_state=42).tolist()
docs_ru = df_ru['body_clean'].sample(min(MAX_DOCS, len(df_ru)), random_state=42).tolist()
print(f'Using {len(docs_en):,} EN and {len(docs_ru):,} RU docs for topic modeling')

## 4. Embeddings (GPU)

`all-MiniLM-L6-v2` is fast and fits comfortably in 8GB VRAM. For Russian, `paraphrase-multilingual-MiniLM-L12-v2` handles Cyrillic well.

In [ ]:
# English embedding model
embed_en = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print('Embedding English docs...')
embeddings_en = embed_en.encode(docs_en, batch_size=512, show_progress_bar=True,
                                 convert_to_numpy=True, normalize_embeddings=True)
print(f'Embeddings shape: {embeddings_en.shape}')

In [ ]:
# Russian / multilingual model
embed_ru = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)
print('Embedding Russian docs...')
embeddings_ru = embed_ru.encode(docs_ru, batch_size=512, show_progress_bar=True,
                                 convert_to_numpy=True, normalize_embeddings=True)
print(f'Embeddings shape: {embeddings_ru.shape}')

## 5. BERTopic — English

In [ ]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=30, metric='euclidean',
                         cluster_selection_method='eom', prediction_data=True)
vectorizer = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))

topic_model_en = BERTopic(
    embedding_model=embed_en,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    top_n_words=10,
    verbose=True,
)

topics_en, probs_en = topic_model_en.fit_transform(docs_en, embeddings_en)
print(f'\nDiscovered {len(set(topics_en)) - 1} topics (excluding outliers)')

In [ ]:
topic_info_en = topic_model_en.get_topic_info()
print(topic_info_en[topic_info_en['Topic'] != -1].head(20).to_string())

In [ ]:
# Visualize top topics
fig = topic_model_en.visualize_barchart(top_n_topics=12, n_words=8, title='Conti EN — top topics')
fig.write_html(str(PROCESSED_DIR / 'bertopic_en_barchart.html'))
fig.show()

## 6. BERTopic — Russian

In [ ]:
umap_ru = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_ru = HDBSCAN(min_cluster_size=30, metric='euclidean',
                      cluster_selection_method='eom', prediction_data=True)
# No stop_words for Russian — use a plain vectorizer
vectorizer_ru = CountVectorizer(min_df=5, ngram_range=(1, 2))

topic_model_ru = BERTopic(
    embedding_model=embed_ru,
    umap_model=umap_ru,
    hdbscan_model=hdbscan_ru,
    vectorizer_model=vectorizer_ru,
    top_n_words=10,
    verbose=True,
)

topics_ru, probs_ru = topic_model_ru.fit_transform(docs_ru, embeddings_ru)
print(f'\nDiscovered {len(set(topics_ru)) - 1} topics (excluding outliers)')

In [ ]:
topic_info_ru = topic_model_ru.get_topic_info()
print(topic_info_ru[topic_info_ru['Topic'] != -1].head(20).to_string())

fig = topic_model_ru.visualize_barchart(top_n_topics=12, n_words=8, title='Conti RU — top topics')
fig.write_html(str(PROCESSED_DIR / 'bertopic_ru_barchart.html'))
fig.show()

## 7. Save topic assignments & models

In [ ]:
# Attach topic labels back to the sampled dataframes
df_en_sample = df_en.sample(min(MAX_DOCS, len(df_en)), random_state=42).copy()
df_en_sample['topic'] = topics_en

df_ru_sample = df_ru.sample(min(MAX_DOCS, len(df_ru)), random_state=42).copy()
df_ru_sample['topic'] = topics_ru

df_en_sample.to_parquet(PROCESSED_DIR / 'conti_topics_en.parquet', index=False)
df_ru_sample.to_parquet(PROCESSED_DIR / 'conti_topics_ru.parquet', index=False)

# Save models for reuse in notebook 03
topic_model_en.save(str(PROCESSED_DIR / 'bertopic_en'))
topic_model_ru.save(str(PROCESSED_DIR / 'bertopic_ru'))

print('Saved topic-labeled datasets and BERTopic models.')